# Five sales measures: inverse of the correlation matrix

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Matrix Inversion** · a worked example from the StatNex teaching guide

**The story.** A sales manager has the correlation matrix of five measures for 20 sales reps: meetings, emails, deals closed, revenue and customer rating. Deals closed and revenue correlate 0.95.

**The question.** What does the inverse of a correlation matrix tell you about variables that overlap?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/matrix_inverse.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Measure,Meetings per week,Emails per day,Deals closed,Revenue ($000),Customer rating
Meetings per week,1.0,0.3586,0.7429,0.7164,0.1357
Emails per day,0.3586,1.0,0.1832,0.1392,-0.2765
Deals closed,0.7429,0.1832,1.0,0.9462,0.149
Revenue ($000),0.7164,0.1392,0.9462,1.0,0.1354
Customer rating,0.1357,-0.2765,0.149,0.1354,1.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The matrix

In [ ]:
# A square matrix, one column per variable. A blank cell counts as 0, as in the app.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Meetings per week", "Emails per day", "Deals closed", "Revenue ($000)", "Customer rating"]
A = data[columns].fillna(0).to_numpy(float)

## Inverse matrix

In [ ]:
# When A is a correlation matrix, each diagonal entry of the inverse is that variable's
# variance inflation factor, VIF = 1 / (1 - R^2), where R^2 comes from regressing the
# variable on all the others. 1 means no overlap; above about 5 to 10 is the usual
# warning (a rule of thumb, not a law) that a regression could not separate it from
# the rest.
inverse = np.linalg.inv(A)
print(pd.DataFrame(inverse, index=columns, columns=columns))

## Determinant

In [ ]:
# det A = 0 means the inverse does not exist; close to 0 means it is unstable. For a
# correlation matrix it lies between 0 and 1: 1 when the variables are uncorrelated,
# near 0 when some of them nearly repeat the others.
determinant = np.linalg.det(A)
print(f"\ndeterminant = {determinant:.6g}")

## Check: A times its inverse

In [ ]:
# Should be the identity matrix; how far it departs shows what the arithmetic lost.
check = A @ inverse
print()
print(pd.DataFrame(check, index=columns, columns=columns))

## Numerical stability

In [ ]:
# The condition number is the largest singular value over the smallest; its
# log10 is roughly how many of the ~16 significant digits are lost.
condition = np.linalg.cond(A)
digits_lost = np.log10(condition)
residual = np.abs(check - np.eye(len(A))).max()
stability = pd.Series({"condition number": condition, "digits lost": digits_lost,
                       "largest departure in the check": residual})
print()
print(stability)

## What the result says

The diagonal of the inverse of a correlation matrix shows how much each variable overlaps with the others. A value near 1 means little overlap. Here the diagonal is 1.17 for customer rating, 1.36 for emails and 2.62 for meetings, but 10.51 for deals closed and 9.82 for revenue.

A value near 10 means the other measures already explain about nine tenths of that variable. The cause is the correlation of 0.95 between deals and revenue. The determinant, 0.034, is close to 0, which is another sign of overlap. The condition number, 52.2, says the inverse can still be computed safely: the check matrix has 1 on its diagonal and almost 0 elsewhere.

So deals and revenue overlap so much that a model using both could not tell them apart. Keep one of them.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.